# Telecom Churn — ML Model Building
Mirrors the original notebook logic exactly: tenure binning, `get_dummies(drop_first=True)`, SMOTEENN, SMOTE, ADASYN, RandomizedSearchCV, Optuna, AdaBoost with sample weights, XGBoost with `scale_pos_weight`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import f1_score, make_scorer
%matplotlib inline

## 1. Load data

In [ ]:
df = pd.read_csv('../artifacts/raw/data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.Churn.value_counts() / len(df) * 100

## **Churn Rate**: ~26.5%
- 26.5% of customers churn — dataset is imbalanced

## 2. Quick baseline (raw data, no cleaning)

In [ ]:
y_raw = df['Churn']
X_raw = df.drop(columns=['customerID', 'Churn'])
X_raw_enc = pd.get_dummies(X_raw, drop_first=True)
y_raw_enc = y_raw.map({'No': 0, 'Yes': 1})
X_tr, X_te, y_tr, y_te = train_test_split(X_raw_enc, y_raw_enc, test_size=0.2)
dt = DecisionTreeClassifier()
dt.fit(X_tr, y_tr)
print(classification_report(y_te, dt.predict(X_te)))

## 3. Data cleaning

In [ ]:
telco = df.copy()
telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce')
print('Nulls before drop:', telco['TotalCharges'].isnull().sum())
telco.dropna(how='any', inplace=True)
print(f'After cleaning: {len(telco)} rows')

## 4. Tenure binning (key feature engineering step)

In [ ]:
bins   = [0, 12, 24, 36, 48, 60, 72]
labels = ['1-12', '13-24', '25-36', '37-48', '49-60', '61-72']
telco['tenure_bin'] = pd.cut(telco['tenure'], bins=bins, labels=labels, include_lowest=True)
print(telco[['tenure', 'tenure_bin']].head(10))
print('\nDistribution:')
print(telco['tenure_bin'].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
telco['tenure_bin'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Customers per tenure slab')
axes[0].set_xlabel('Tenure slab')
telco.groupby('tenure_bin')['Churn'].apply(lambda x: (x=='Yes').mean() if x.dtype==object else x.mean()).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Churn rate per tenure slab')
axes[1].set_xlabel('Tenure slab')
plt.tight_layout()
plt.show()

## 5. Feature engineering — X and y split

In [ ]:
# Drop customerID, raw tenure (replaced by tenure_bin), and target
y = telco['Churn'].map({'No': 0, 'Yes': 1})
X = telco.drop(columns=['customerID', 'Churn', 'tenure'])
print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

In [ ]:
# One-hot encode with drop_first=True (same as notebook)
X = pd.get_dummies(X, drop_first=True)
bool_cols = X.select_dtypes(include='bool').columns
X[bool_cols] = X[bool_cols].astype(int)
print('X shape after encoding:', X.shape)
print('Columns:', X.columns.tolist())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 6. Feature scaling — StandardScaler

In [ ]:
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc  = sc.transform(X_test)

dt2 = DecisionTreeClassifier()
dt2.fit(X_train_sc, y_train)
print('DT + StandardScaler')
print(classification_report(y_test, dt2.predict(X_test_sc)))

## 7. Feature scaling — MinMaxScaler

In [ ]:
mms = MinMaxScaler()
X_train_mms = mms.fit_transform(X_train)
X_test_mms  = mms.transform(X_test)

dt3 = DecisionTreeClassifier()
dt3.fit(X_train_mms, y_train)
print('DT + MinMaxScaler')
print(classification_report(y_test, dt3.predict(X_test_mms)))

## 8. SMOTEENN — upsample + ENN clean

In [ ]:
from imblearn.combine import SMOTEENN
from collections import Counter

sm = SMOTEENN(random_state=42)
X_train_smoteenn, y_train_smoteenn = sm.fit_resample(X_train_sc, y_train)
print('Before SMOTEENN:', Counter(y_train))
print('After  SMOTEENN:', Counter(y_train_smoteenn))

In [ ]:
dt_sm = DecisionTreeClassifier()
dt_sm.fit(X_train_smoteenn, y_train_smoteenn)
print('DT + SMOTEENN')
print(classification_report(y_test, dt_sm.predict(X_test_sc)))

In [ ]:
rf_sm = RandomForestClassifier(n_estimators=500, random_state=42)
rf_sm.fit(X_train_smoteenn, y_train_smoteenn)
print('RandomForest (500) + SMOTEENN')
print(classification_report(y_test, rf_sm.predict(X_test_sc)))

## 9. XGBoost experiments

In [ ]:
from xgboost import XGBClassifier

# 9a — no resampling
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train_sc, y_train)
print('XGBoost — no resampling')
print(classification_report(y_test, xgb.predict(X_test_sc)))

In [ ]:
# 9b — with SMOTEENN
xgb_sm = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_sm.fit(X_train_smoteenn, y_train_smoteenn)
print('XGBoost + SMOTEENN')
print(classification_report(y_test, xgb_sm.predict(X_test_sc)))

In [ ]:
# 9c — with SMOTE
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_sc, y_train)
print('Before SMOTE:', Counter(y_train))
print('After  SMOTE:', Counter(y_train_smote))

xgb_smote = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_smote.fit(X_train_smote, y_train_smote)
print('XGBoost + SMOTE')
print(classification_report(y_test, xgb_smote.predict(X_test_sc)))

In [ ]:
# 9d — ADASYN
from imblearn.over_sampling import ADASYN
adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_sc, y_train)
print('Before ADASYN:', Counter(y_train))
print('After  ADASYN:', Counter(y_train_ad))

xgb_ad = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_ad.fit(X_train_ad, y_train_ad)
print('XGBoost + ADASYN')
print(classification_report(y_test, xgb_ad.predict(X_test_sc)))

In [ ]:
# 9e — scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

xgb_w = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')
xgb_w.fit(X_train_sc, y_train)
print('XGBoost + scale_pos_weight')
print(classification_report(y_test, xgb_w.predict(X_test_sc)))

## 10. AdaBoost with sample weighting

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
weight_positive = negative_count / positive_count
print(f'Neg: {negative_count}, Pos: {positive_count}, weight_pos: {weight_positive:.2f}')

sample_weight = np.where(y_train == 1, weight_positive, 1.0)

model_ada_weighted = AdaBoostClassifier(n_estimators=50, random_state=42)
model_ada_weighted.fit(X_train_sc, y_train, sample_weight=sample_weight)

y_pred_ada = model_ada_weighted.predict(X_test_sc)
print('AdaBoost with sample weighting')
print(classification_report(y_test, y_pred_ada))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred_ada))

## 11. Hyperparameter tuning — RandomizedSearchCV (XGBoost)

In [ ]:
param_grid = {
    'max_depth':        [3, 4, 5, 6, 7],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'n_estimators':     [100, 200, 300, 400],
    'subsample':        [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':            [0, 0.1, 0.2],
    'scale_pos_weight': [1, scale_pos_weight],
}

xgb_rs = XGBClassifier(random_state=42, eval_metric='logloss')
search = RandomizedSearchCV(
    xgb_rs, param_grid, n_iter=50, cv=5,
    scoring='f1', random_state=42, n_jobs=-1
)
search.fit(X_train_sc, y_train)

print('Best params:', search.best_params_)
print('Best CV F1 :', search.best_score_)

best_xgb = search.best_estimator_
print('\nTest report:')
print(classification_report(y_test, best_xgb.predict(X_test_sc)))

In [ ]:
import joblib
joblib.dump(best_xgb, '../artifacts/models/best_xgboost_randomsearch.pkl')
print('Saved best_xgboost_randomsearch.pkl')

## 12. Optuna fine-tuning (XGBoost)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'max_depth':          trial.suggest_int('max_depth', 3, 10),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators':       trial.suggest_int('n_estimators', 100, 1000),
        'subsample':          trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':   trial.suggest_int('min_child_weight', 1, 10),
        'gamma':              trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha':          trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':         trial.suggest_float('reg_lambda', 0.0, 2.0),
        'scale_pos_weight':   trial.suggest_categorical('scale_pos_weight', [1, scale_pos_weight]),
        'random_state':       42,
        'eval_metric':        'logloss',
    }
    model = XGBClassifier(**params)
    f1_scorer = make_scorer(f1_score, average='binary', pos_label=1)
    return cross_val_score(model, X_train_sc, y_train, cv=5, scoring=f1_scorer, n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print('Best params:', study.best_params)
print('Best CV F1 :', study.best_value)

best_optuna = XGBClassifier(**{**study.best_params, 'eval_metric': 'logloss', 'random_state': 42})
best_optuna.fit(X_train_sc, y_train)
print('\nTest report:')
print(classification_report(y_test, best_optuna.predict(X_test_sc)))

In [ ]:
joblib.dump(best_optuna, '../artifacts/models/best_xgboost_optuna.pkl')
print('Saved best_xgboost_optuna.pkl')

## 13. Save production models (AdaBoost + best XGBoost)

In [ ]:
# AdaBoost — production model
joblib.dump(model_ada_weighted, '../artifacts/models/adaboost_model.pkl')
print('Saved adaboost_model.pkl')

# XGBoost — use Optuna best
joblib.dump(best_optuna, '../artifacts/models/xgboost_model.pkl')
print('Saved xgboost_model.pkl')

## 14. ROC curves comparison

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fig, ax = plt.subplots(figsize=(8, 6))
for name, model, color in [
    ('AdaBoost (sample weighted)', model_ada_weighted, '#e74c3c'),
    ('XGBoost (Optuna)',           best_optuna,        '#3498db'),
    ('XGBoost (RandomSearch)',     best_xgb,           '#2ecc71'),
]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test_sc)[:,1])
    auc = roc_auc_score(y_test, model.predict_proba(X_test_sc)[:,1])
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend()
plt.tight_layout()
plt.show()